# Result visualisation

This notebook plots the experimental result arrays stored below.


In [ ]:
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

In [ ]:
def process_data(data_file):
    data_different_row = [row.split() for row in data_file.strip().split('\n')]
    data_matrix = np.expand_dims(np.array(data_different_row, dtype=float), axis=-1)
    data_matrix = np.concatenate((data_matrix[:, :5], data_matrix[:, 5:10], data_matrix[:, 10:15], data_matrix[:, 15:20], data_matrix[:, 20:25], data_matrix[:, 25:30]), axis=-1)

    return data_matrix

In [ ]:

amazon_semi_synthetic = """
0.8773748448	0.8829888766	0.8794983054	0.8798747729	0.8798738269	0.9674991042	0.935236849	0.9777494437	0.9684999921	0.9789993647	0.9814996658	0.9796239656	0.983749999	0.9892499456	0.9948749999	0.9751247571	0.9721249961	0.9784995646	0.9608638554	0.9752497386	0.9504925708	0.9816242532	0.9853749486	0.9796226996	0.9821248992	0.8998749984	0.9011249243	0.9001249984	0.9302996703	0.9
0.8747465477	0.8762282703	0.8807408692	0.8784989065	0.8849971249	0.9217253386	0.9202326456	0.9212446397	0.926249221	0.9302463336	0.9252277506	0.9267415341	0.9159931954	0.9264995406	0.9242465485	0.9152466893	0.9164969939	0.9199975799	0.9159810957	0.9234990629	0.9224788949	0.9244945447	0.9214929147	0.9206797967	0.9229898154	0.9	0.8997499937	0.9	0.8916508538	0.9
0.7044993351	0.6987010201	0.7054910911	0.703999704	0.7084953359	0.8085383272	0.7827775642	0.8399686339	0.8199819982	0.829157649	0.845130741	0.8191644535	0.8352445528	0.8232486633	0.8352232135	0.8349930285	0.7924771206	0.8159403647	0.7841965264	0.8139865605	0.8118132717	0.8252193151	0.8292302654	0.8300836625	0.820917219	0.7	0.7	0.7	0.7231449159	0.7
0.5317476295	0.5214612384	0.5304957745	0.5252343031	0.5312476269	0.6935365339	0.64898283	0.7578880838	0.7179658739	0.7199587396	0.7646327683	0.7189202759	0.7604496345	0.7272479373	0.7411936935	0.7562371872	0.671634559	0.7128345927	0.6485890341	0.6987118682	0.6981042261	0.7331237509	0.7403685616	0.7439501469	0.7154540544	0.5	0.5002499688	0.5	0.5529437974	0.5
0.3577114225	0.3464867664	0.355999356	0.3514531425	0.35122855	0.5779614483	0.5153128039	0.6750086736	0.6149575491	0.6153051449	0.6871782988	0.6150790587	0.6821942052	0.6302377747	0.6471192356	0.6774863738	0.5502300256	0.6071601264	0.517646762	0.5859403754	0.5899756186	0.6377736085	0.6522175137	0.6576651249	0.6126532682	0.3	0.3002499563	0.3	0.3852690476	0.3
0.1895967831	0.1739981415	0.1822499489	0.1741050038	0.1716629728	0.4675158202	0.38215735	0.5878213084	0.5177130749	0.5114918166	0.6109749461	0.5138715524	0.6041518049	0.534448673	0.5565856873	0.6052345795	0.4296568074	0.5020908363	0.3830500416	0.47987504	0.4842045171	0.5437455822	0.560915029	0.5723251341	0.5089092917	0.1	0.1007499438	0.1	0.2192406844	0.1
0.542972785	0.503712812	0.5285840095	0.5148602649	0.5178722478	0.8229934097	0.8615630322	0.8658494495	0.8623011629	0.8140398035	0.8654367283	0.8729140899	0.8548272081	0.858169999	0.8289783948	0.8672248972	0.8652566198	0.8641599295	0.8544358062	0.8535071982	0.8487967588	0.8683854624	0.8556879613	0.8659490271	0.7660481097	0.4984580065	0.6380208333	0.5730233873	0.7083651918	0.4813259058
"""


semi_synthetic_amazon_matrix = process_data(amazon_semi_synthetic)
print(f'create matrix of shape {semi_synthetic_amazon_matrix.shape}')
print(f'Synthetic mean average results is \n {np.mean(semi_synthetic_amazon_matrix, axis=1)*100}')
print(f'Synthetic std average results is \n {np.std(semi_synthetic_amazon_matrix, axis=1)*100}')

In [ ]:
def get_p_matrices(matrix):
    N, T, s = matrix.shape
    p_matrices = np.zeros((N, s, s))

    # Perform t-tests for each case
    for n in range(N):
        for i in range(s):
            for j in range(i, s):
                if i == j:
                    # Diagonal should be zero since it is the same method
                    p_matrices[n, i, j] = 0.0
                else:
                    # Extract readings for the current case
                    readings_i = matrix[n, :, i]
                    readings_j = matrix[n, :, j]

                    # Perform the t-test
                    t_stat, p_value = ttest_ind(readings_i, readings_j, equal_var=False)

                    if p_value < 0.001:
                        p_value = -0.001
                    elif p_value < 0.05:
                        p_value = -0.05

                    # Store the p-value in the matrix
                    p_matrices[n, i, j] = p_value
                    p_matrices[n, j, i] = p_value  # Symmetric matrix
    
    return p_matrices

In [ ]:
p_matrix = get_p_matrices(semi_synthetic_amazon_matrix)
for matrix in p_matrix:
    print('------')
    for line in matrix:
        print(line)
# p_matrix.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = ["SFT", "CTL", "CTL-N", "CTL-C"]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 30%", "OOD 10%"]

custom_palette = {
    'SFT': sns.color_palette("Set3")[0],
    'CTL': sns.color_palette("Set3")[1],
    'CTL-N': sns.color_palette("Set3")[2],
    'CTL-C': sns.color_palette("Set3")[3],
}

color_list = [custom_palette[model] for model in model_names]

# Exclude the OOD 0% case
amazon_vis_matrix = semi_synthetic_amazon_matrix[1:6, :, 1:-1]
N, T, s = amazon_vis_matrix.shape

# Create a figure with subplots (1 row, 5 columns)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
sns.set(style="whitegrid")

margin = 0.005

for n in range(N):
    # Prepare data for plotting
    plot_data = [amazon_vis_matrix[n, :, method] for method in range(s)]
    
    # Find min and max for this specific case
    case_min = np.min(amazon_vis_matrix[n]) - margin
    case_max = np.max(amazon_vis_matrix[n]) + margin
    
    # Create box plot for each case
    sns.boxplot(data=plot_data, palette=color_list, width=0.6, ax=axes[n])
    
    # Customize each subplot
    axes[n].set_ylabel('F1 Score', fontsize=12)
    axes[n].set_title(case_labels[n], fontsize=14)
    axes[n].set_xticklabels(model_names, rotation=45, ha='right', fontsize=10)
    axes[n].tick_params(axis='y', labelsize=10)
    axes[n].grid(True, linestyle='--', alpha=0.7)
    
    # Set y-axis limits to case-specific min and max
    axes[n].set_ylim(case_min, case_max)

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:


yelp_semi_synthetic = """
0.8629037241	0.8574150236	0.8629055333	0.8654501409	0.8634661311	0.9372209804	0.964499199	0.9769999942	0.9706242284	0.9484919519	0.978497956	0.9828749398	0.9871249982	0.9921249851	0.9941249793	0.9574550619	0.9832499832	0.9824997856	0.991749967	0.9748688645	0.9782463236	0.9851248975	0.9882499882	0.9932499895	0.9862495454	0.9175676567	0.9099942396	0.9114965428	0.9525768173	0.9294219716
0.8659195182	0.855074994	0.8634076636	0.8669592563	0.8697134465	0.924465084	0.9277483698	0.931749791	0.9297385781	0.9307187152	0.9331783296	0.9287393064	0.9307492685	0.9264787524	0.9322492844	0.9122894069	0.9272486859	0.9272341689	0.9269866933	0.9236824542	0.927156902	0.9339957757	0.9332499958	0.9277353258	0.9271855547	0.8964751281	0.8997485902	0.8934846618	0.8789066322	0.8961938863
0.7185486039	0.7139337359	0.7147997181	0.7141649462	0.7174558525	0.8013250218	0.8014939952	0.8419898874	0.8178572	0.8319709773	0.8376408997	0.8334040407	0.84074999	0.8424129726	0.8539894507	0.7859653329	0.8319965979	0.8189476759	0.843409804	0.8151858432	0.8271161747	0.8467338975	0.837241445	0.8479195494	0.8443569917	0.7174828624	0.7074835459	0.7039499675	0.7362565362	0.7262139097
0.5729777466	0.5642682713	0.5648250244	0.5672999844	0.5713833091	0.6799857414	0.6797195533	0.7572135669	0.7122603931	0.7306093271	0.7505462156	0.7442698429	0.754997795	0.768875564	0.77293046	0.6622859061	0.7427364776	0.7145301704	0.768875564	0.7059725859	0.7313372348	0.7594561609	0.7477089869	0.7741908673	0.763030064	0.5392644579	0.5168913005	0.5151372391	0.5958085013	0.5535255785
0.4238529987	0.4180673566	0.4182691631	0.4182673069	0.4235977938	0.5497195212	0.5533387553	0.6761037406	0.598291774	0.6339409177	0.654469735	0.6468477375	0.6719211791	0.6872427457	0.6926239566	0.5286278387	0.6481834847	0.6118948595	0.6888421818	0.5955628368	0.6261872629	0.6685784686	0.6602430588	0.6939617794	0.681923581	0.3564137417	0.3269315966	0.3268276679	0.4513587634	0.3840611268
0.2715447154	0.2757178527	0.2651212653	0.2632612967	0.2711657741	0.4240721714	0.4325056582	0.5886775232	0.4896956396	0.5268637244	0.5619725767	0.5555118997	0.5852510469	0.6098260562	0.6072642604	0.3905818235	0.55961125	0.5099982382	0.6136486106	0.4794310512	0.5198323095	0.58535793	0.5737203209	0.6148285005	0.593997645	0.1748585795	0.1346983161	0.1384471103	0.2990727952	0.2069220691
0.5997633115	0.5808728487	0.5872887086	0.5959038468	0.5987131926	0.8324810777	0.8439494396	0.8221131724	0.7776842105	0.8310053176	0.8241128152	0.8482213043	0.8444018808	0.8294651636	0.8258405687	0.8261916984	0.8437112306	0.8046214655	0.8478958921	0.8347118977	0.8004400949	0.8300419106	0.8468945135	0.8415851658	0.8498590852	0.7037157607	0.7109732559	0.4664994379	0.7301204819	0.6649078289
"""

semi_synthetic_yelp_matrix = process_data(yelp_semi_synthetic)
print(f'create matrix of shape {semi_synthetic_yelp_matrix.shape}')
print(f'Synthetic mean average results is \n {np.mean(semi_synthetic_yelp_matrix, axis=1)*100}')
print(f'Synthetic std average results is \n {np.std(semi_synthetic_yelp_matrix, axis=1)*100}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = ["SFT", "CTL", "CTL-N", "CTL-C"]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 30%", "OOD 10%"]

custom_palette = {
    'SFT': sns.color_palette("Set3")[0],
    'CTL': sns.color_palette("Set3")[1],
    'CTL-N': sns.color_palette("Set3")[2],
    'CTL-C': sns.color_palette("Set3")[3],
}

color_list = [custom_palette[model] for model in model_names]

# Exclude the OOD 0% case
amazon_vis_matrix = semi_synthetic_yelp_matrix[1:6, :, 1:-1]
N, T, s = amazon_vis_matrix.shape

# Create a figure with subplots (2 rows, 3 columns)
fig, axes = plt.subplots(2, 3, figsize=(25, 10))
sns.set(style="whitegrid")

margin = 0.005

for row in range(2):
    for col in range(3):
        n = row * 3 + col
        if n < N:
            ax = axes[row, col]
            
            # Prepare data for plotting
            plot_data = [amazon_vis_matrix[n, :, method] for method in range(s)]
            
            # Find min and max for this specific case
            case_min = np.min(amazon_vis_matrix[n]) - margin
            case_max = np.max(amazon_vis_matrix[n]) + margin
            
            # Create box plot for each case
            sns.boxplot(data=plot_data, palette=color_list, width=0.6, ax=ax)
            
            # Customize each subplot
            ax.set_ylabel('F1 Score', fontsize=12)
            ax.set_title(case_labels[n], fontsize=14)
            ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=10)
            ax.tick_params(axis='y', labelsize=10)
            ax.grid(True, linestyle='--', alpha=0.7)
            
            # Set y-axis limits to case-specific min and max
            ax.set_ylim(case_min, case_max)
        else:
            # Remove the extra subplot
            fig.delaxes(axes[row, col])

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
real_data_sentiment = """
0.8776236061	0.877243985	0.8783585613	0.8779997255	0.8757486876	0.948499456	0.9479845221	0.9208178094	0.9208736536	0.9624999789	0.9726249996	0.995249976	0.96599285	0.9664941176	0.972872982	0.8997499937	0.8992499748	0.90987074	0.9526203797	0.9066243566	0.938624953	0.920998573	0.970121302	0.9694933629	0.9806249755	0.9159680468	0.9283644685	0.9003748116	0.9002499751	0.9011249861
0.877999878	0.8775	0.8772459414	0.878498056	0.8779992375	0.9122334066	0.9149768524	0.9104220448	0.9097483698	0.921999922	0.9102365299	0.9034594288	0.899065536	0.9069528949	0.9096924233	0.8982499428	0.8967496838	0.8989993687	0.9049851539	0.9002496945	0.9142490943	0.9144982686	0.9033883169	0.9046712048	0.9167443334	0.8982422092	0.8992383557	0.8974999744	0.8972498395	0.8985
0.6972499811	0.6934972415	0.6947360914	0.6967267806	0.6964996965	0.7764874274	0.7981899489	0.7701587235	0.7501956519	0.8076960467	0.8009928357	0.8134842033	0.7931635022	0.8123130531	0.7960695088	0.6992498308	0.697997282	0.7032433044	0.7607394453	0.7044952719	0.7832200708	0.7534997535	0.7900161973	0.8169258796	0.802999212	0.7089056854	0.718363688	0.6987499812	0.6982498303	0.7
0.5164995165	0.5104851922	0.515479529	0.5149562248	0.515748517	0.6437337676	0.6891287408	0.6289185935	0.5886540581	0.6872886071	0.7047485053	0.7239224403	0.690129574	0.7072869702	0.6777278991	0.5002497189	0.5002462206	0.507716494	0.6159514201	0.5087276164	0.6561416631	0.5982499749	0.6765430647	0.7188967149	0.6902498258	0.5221118353	0.5382547494	0.5002499688	0.4999995	0.5012499688
0.3342449652	0.3309759151	0.3341930319	0.3366932458	0.3349940149	0.5107448322	0.5714767897	0.4875570055	0.4318522911	0.5665968267	0.6029991067	0.6283812286	0.5928841572	0.6005881674	0.5592238456	0.3007489074	0.3017464651	0.3121777357	0.4706756314	0.3124613259	0.5281167128	0.4392491238	0.5672550642	0.625209018	0.5709870224	0.3364383013	0.3581578655	0.3024998256	0.3017496072	0.3027499564
0.151999152	0.1509864158	0.1499455965	0.1551333556	0.1549864798	0.37974686	0.450068854	0.3449651946	0.2681704261	0.4459601606	0.5027119264	0.52884978	0.4936931394	0.4915550189	0.4440539844	0.1017494947	0.102996412	0.1153617753	0.3169685	0.115665933	0.3977251918	0.2777495937	0.4554466051	0.5249361196	0.4566472724	0.1490148343	0.1712804963	0.102999103	0.1027499439	0.103999104
"""
real_matrix = process_data(real_data_sentiment)
print(f'create matrix of shape {real_matrix.shape}')
print(f'Synthetic mean average results is \n {np.mean(real_matrix, axis=1)*100}')
print(f'Synthetic std average results is \n {np.std(real_matrix, axis=1)*100}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = ["SFT", "CTL", "CTL-N", "CTL-C"]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 30%", "OOD 10%"]

custom_palette = {
    'SFT': sns.color_palette("Set3")[0],
    'CTL': sns.color_palette("Set3")[1],
    'CTL-N': sns.color_palette("Set3")[2],
    'CTL-C': sns.color_palette("Set3")[3],
}

color_list = [custom_palette[model] for model in model_names]

amazon_vis_matrix = real_matrix[1:, :, 1:-1]
N, T, s = amazon_vis_matrix.shape

# Calculate the number of rows and columns
n_rows = 2
n_cols = (N + 1) // 2  # Ceiling division to handle odd N

# Create a figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
sns.set(style="whitegrid")

# Flatten the axes array for easier iteration
axes_flat = axes.flatten()

margin = 0.005

for n in range(N):
    # Prepare data for plotting
    plot_data = [amazon_vis_matrix[n, :, method] for method in range(s)]
    
    # Find min and max for this specific case
    case_min = np.min(amazon_vis_matrix[n]) - margin
    case_max = np.max(amazon_vis_matrix[n]) + margin
    
    # Create box plot for each case
    sns.boxplot(data=plot_data, palette=color_list, width=0.6, ax=axes_flat[n])
    
    # Customize each subplot
    axes_flat[n].set_ylabel('F1 Score', fontsize=12)
    axes_flat[n].set_title(case_labels[n], fontsize=14)
    axes_flat[n].set_xticklabels(model_names, rotation=45, ha='right', fontsize=10)
    axes_flat[n].tick_params(axis='y', labelsize=10)
    axes_flat[n].grid(True, linestyle='--', alpha=0.7)
    
    # Set y-axis limits to case-specific min and max
    axes_flat[n].set_ylim(case_min, case_max)

# Remove any extra subplots
for i in range(N, len(axes_flat)):
    fig.delaxes(axes_flat[i])

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
real_with_baeline = '''
0.948499456	0.9479845221	0.9208178094	0.9208736536	0.9624999789	1	0.999875	1	1	1	0.9278648278	0.9277342777	0.9326150934	0.920624969	0.9348665723	0.9726249996	0.995249976	0.96599285	0.9664941176	0.972872982	0.8997499937	0.8992499748	0.90987074	0.9526203797	0.9066243566	0.938624953	0.920998573	0.970121302	0.9694933629	0.9806249755
0.9122334066	0.9149768524	0.9104220448	0.9097483698	0.921999922	0.9137499515	0.9152471979	0.914499658	0.9054971413	0.9139944956	0.9134913491	0.9164898953	0.9162329901	0.9074999075	0.9132408849	0.9102365299	0.9034594288	0.899065536	0.9069528949	0.9096924233	0.8982499428	0.8967496838	0.8989993687	0.9049851539	0.9002496945	0.9142490943	0.9144982686	0.9033883169	0.9046712048	0.9167443334
0.7764874274	0.7981899489	0.7701587235	0.7501956519	0.8076960467	0.8004712679	0.807930663	0.8012445218	0.8048922517	0.8023861744	0.7537556641	0.7825238112	0.7804849218	0.7382380735	0.7744873149	0.8009928357	0.8134842033	0.7931635022	0.8123130531	0.7960695088	0.6992498308	0.697997282	0.7032433044	0.7607394453	0.7044952719	0.7832200708	0.7534997535	0.7900161973	0.8169258796	0.802999212
0.6437337676	0.6891287408	0.6289185935	0.5886540581	0.6872886071	0.6978909386	0.70650741	0.6908292605	0.6977603529	0.6887582185	0.5916169738	0.6551378446	0.6406408137	0.5649207068	0.6362039571	0.7047485053	0.7239224403	0.690129574	0.7072869702	0.6777278991	0.5002497189	0.5002462206	0.507716494	0.6159514201	0.5087276164	0.6561416631	0.5982499749	0.6765430647	0.7188967149	0.6902498258
0.5107448322	0.5714767897	0.4875570055	0.4318522911	0.5665968267	0.5912981546	0.5968868649	0.5794261318	0.5872488463	0.5748724555	0.43039865	0.520378996	0.505062572	0.3932809744	0.4988872496	0.6029991067	0.6283812286	0.5928841572	0.6005881674	0.5592238456	0.3007489074	0.3017464651	0.3121777357	0.4706756314	0.3124613259	0.5281167128	0.4392491238	0.5672550642	0.625209018	0.5709870224
0.37974686	0.450068854	0.3449651946	0.2681704261	0.4459601606	0.4792239685	0.4847880928	0.4676482372	0.4778917761	0.4608201413	0.2703516009	0.3761503644	0.3696597533	0.2183454449	0.3571371374	0.5027119264	0.52884978	0.4936931394	0.4915550189	0.4440539844	0.1017494947	0.102996412	0.1153617753	0.3169685	0.115665933	0.3977251918	0.2777495937	0.4554466051	0.5249361196	0.4566472724
'''

real_with_baseline_matrix = process_data(real_with_baeline)
print(f'create matrix of shape {real_with_baseline_matrix.shape}')
print(f'Synthetic mean average results is \n {np.mean(real_with_baseline_matrix, axis=1)*100}')
# print(f'Synthetic std average results is \n {np.std(real_with_baseline_matrix, axis=1)*100}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = ["SFT", "SWA", "WISE", "CTL", "CTL-N", "CTL-C"]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 30%", "OOD 10%"]

# custom_palette = {
#     'SFT': sns.color_palette("Set3")[0],
#     'CTL': sns.color_palette("Set3")[1],
#     'CTL-N': sns.color_palette("Set3")[2],
#     'CTL-C': sns.color_palette("Set3")[3],
# }

custom_palette = {
    'SFT': sns.color_palette("Set3")[0],
    'CTL': sns.color_palette("Set3")[1],
    'CTL-N': sns.color_palette("Set3")[2],
    'CTL-C': sns.color_palette("Set3")[3],
    'SWA': sns.color_palette("Set3")[4],
    'WISE': sns.color_palette("Set3")[5],
}

color_list = [custom_palette[model] for model in model_names]

# Exclude the OOD 0% case
amazon_vis_matrix = real_with_baseline_matrix[1:, :, :]
N, T, s = amazon_vis_matrix.shape

# Create a figure with subplots (1 row, 5 columns)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
sns.set(style="whitegrid")

margin = 0.005

for n in range(N):
    # Prepare data for plotting
    plot_data = [amazon_vis_matrix[n, :, method] for method in range(s)]
    
    # Find min and max for this specific case
    case_min = np.min(amazon_vis_matrix[n]) - margin
    case_max = np.max(amazon_vis_matrix[n]) + margin
    
    # Create box plot for each case
    sns.boxplot(data=plot_data, palette=color_list, width=0.6, ax=axes[n])
    
    # Customize each subplot
    axes[n].set_ylabel('F1 Score', fontsize=12)
    axes[n].set_title(case_labels[n], fontsize=14)
    axes[n].set_xticklabels(model_names, rotation=45, ha='right', fontsize=10)
    axes[n].tick_params(axis='y', labelsize=10)
    axes[n].grid(True, linestyle='--', alpha=0.7)
    
    # Set y-axis limits to case-specific min and max
    axes[n].set_ylim(case_min, case_max)

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = ["SFT", "SWA", "WISE", "CTL", "CTL-N", "CTL-C"]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 10%"]

custom_palette = {
    'SFT': sns.color_palette("Set3")[0],
    'CTL': sns.color_palette("Set3")[1],
    'CTL-N': sns.color_palette("Set3")[2],
    'CTL-C': sns.color_palette("Set3")[3],
    'SWA': sns.color_palette("Set3")[4],
    'WISE': sns.color_palette("Set3")[5],
}

color_list = [custom_palette[model] for model in model_names]

# Select only the specified cases
selected_cases = [1, 2, 4, 5]  # Indices for 90%, OOD 70%, OOD 50%, and OOD 10%
amazon_vis_matrix = real_with_baseline_matrix[selected_cases]

N, T, s = amazon_vis_matrix.shape

# Create a figure with subplots
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
sns.set(style="whitegrid")

margin = 0.005

for n, ax in enumerate(axes):
    # Prepare data for plotting
    plot_data = [amazon_vis_matrix[n, :, method] for method in range(s)]
    
    # Find min and max for this specific case
    case_min = np.min(amazon_vis_matrix[n]) - margin
    case_max = np.max(amazon_vis_matrix[n]) + margin
    
    # Create box plot for each case
    sns.boxplot(data=plot_data, palette=color_list, width=0.6, ax=ax)
    
    # Customize each subplot
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title(case_labels[n], fontsize=14)
    ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Set y-axis limits to case-specific min and max
    ax.set_ylim(case_min, case_max)

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
synthetic_different_spurious_level ='''
0.8773748448	0.8829888766	0.8794983054	0.9674991042	0.935236849	0.9777494437	0.9814996658	0.9796239656	0.983749999	0.8876239026	0.8866243605	0.8878749562	0.9578743675	0.947605836	0.9597484277	0.9672446752	0.9683749165	0.9706235087	0.8906231389	0.8928748644	0.8927491889	0.9478693886	0.9504999505	0.953999264	0.9606070017	0.9629977776	0.9587391045			
0.8747465477	0.8762282703	0.8807408692	0.9217253386	0.9202326456	0.9212446397	0.9252277506	0.9267415341	0.9159931954	0.8857499929	0.8899966724	0.88999901	0.9187323192	0.9172108356	0.9142165391	0.9049254616	0.911213592	0.9124731949	0.8924978231	0.894999895	0.894248883	0.9164994781	0.9147427052	0.9169837288	0.8998190482	0.9097487308	0.8982940454			
0.7044993351	0.6987010201	0.7054910911	0.8085383272	0.7827775642	0.8399686339	0.845130741	0.8191644535	0.8352445528	0.7014999254	0.6989908945	0.7017498322	0.7908301018	0.7818227954	0.7858677606	0.8010222544	0.8094077534	0.7899028826	0.7002270486	0.700998804	0.6982400231	0.7742411813	0.7782266999	0.7830243317	0.7936548101	0.7824129652	0.7926283906			
0.5317476295	0.5214612384	0.5304957745	0.6935365339	0.64898283	0.7578880838	0.7646327683	0.7189202759	0.7604496345	0.510998044	0.5092364733	0.5132497262	0.6611306952	0.6444593653	0.6614191653	0.693203786	0.7142993667	0.6651463108	0.5054345937	0.5059920959	0.5047202523	0.625221306	0.6439144504	0.6473779747	0.6809036647	0.6594485907	0.693444214			
0.3577114225	0.3464867664	0.355999356	0.5779614483	0.5153128039	0.6750086736	0.6871782988	0.6150790587	0.6821942052	0.3252347756	0.3202428201	0.3234993235	0.5327184555	0.5079264512	0.5324047293	0.5887965052	0.6176759165	0.5407462723	0.3131294113	0.310731002	0.3091843293	0.4822147582	0.5104253814	0.5084585259	0.5754964418	0.5286971323	0.5870584106			
0.1895967831	0.1739981415	0.1822499489	0.4675158202	0.38215735	0.5878213084	0.6109749461	0.5138715524	0.6041518049	0.137482534	0.1314893607	0.133996536	0.4128279498	0.3729134543	0.3997453832	0.4932425281	0.5245156285	0.4172385625	0.1195584489	0.1152034929	0.1158830755	0.3432243447	0.3786432063	0.3696157291	0.4685245465	0.4062089218	0.4790780036			
0.542972785	0.503712812	0.5285840095	0.8229934097	0.8615630322	0.8658494495	0.8654367283	0.8729140899	0.8548272081	0.5013754384	0.4832425027	0.5023074739	0.8527260588	0.8727482105	0.8692252754	0.8612724521	0.8696795537	0.847539831	0.4993984707	0.4981250789	0.5001904762	0.8468825923	0.8630750013	0.8597812237	0.8610485467	0.8677267778	0.839947127			
'''

def process_data_spurious(data_file):
    data_different_row = [row.split() for row in data_file.strip().split('\n')]
    data_matrix = np.expand_dims(np.array(data_different_row, dtype=float), axis=-1)
    data_matrix = np.concatenate((data_matrix[:, :3], data_matrix[:, 3:6], data_matrix[:, 6:9], data_matrix[:, 9:12], data_matrix[:, 12:15], data_matrix[:, 15:18], data_matrix[:, 18:21], data_matrix[:, 21:24], data_matrix[:, 24:27]), axis=-1)

    return data_matrix

synthetic_spurious_matrix = process_data_spurious(synthetic_different_spurious_level)
print(f'create matrix of shape {synthetic_spurious_matrix.shape}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = [
    ["SFT-1", "CTL-1"],
    ["SFT-2", "CTL-2"],
    ["SFT-3", "CTL-3"]
]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 10%"]

custom_palette = {
    'SFT': sns.color_palette("Set3")[1],
    'CTL': sns.color_palette("Set3")[2],
}

# Select only the specified cases
selected_cases = [1, 2, 4, 5]  # Indices for 90%, OOD 70%, OOD 50%, and OOD 10%
amazon_vis_matrix = synthetic_spurious_matrix[selected_cases]

N, T, s = amazon_vis_matrix.shape

# Create a figure with subplots
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
sns.set(style="whitegrid")

margin = 0.005

for row in range(3):
    for col, case in enumerate(selected_cases):
        ax = axes[row, col]
        
        # Prepare data for plotting, skipping SFT0
        plot_data = [amazon_vis_matrix[col, :, 3*row + method + 1] for method in range(2)]
        
        # Find min and max for this specific case
        case_min = np.min(amazon_vis_matrix[col, :, 3*row+1:3*row+3]) - margin
        case_max = np.max(amazon_vis_matrix[col, :, 3*row+1:3*row+3]) + margin
        
        # Create box plot for each case
        sns.boxplot(data=plot_data, palette=[custom_palette[model.split('-')[0]] for model in model_names[row]], width=0.6, ax=ax)
        
        # Customize each subplot
        ax.set_ylabel('F1 Score', fontsize=12)
        if row == 0:
            ax.set_title(case_labels[col], fontsize=14)
        ax.set_xticklabels(model_names[row], rotation=45, ha='right', fontsize=10)
        ax.tick_params(axis='y', labelsize=10)
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # Set y-axis limits to case-specific min and max
        ax.set_ylim(case_min, case_max)

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
different_training_size ='''
0.9396308804	0.9740615856	0.9481245897	0.9715573642	0.9778115444	0.9960937453	0.9674991042	0.935236849	0.9777494437	0.9814996658	0.9796239656	0.983749999	0.9714738068	0.9337344048	0.9482754641	0.9813636325	0.9849999876	0.9852271964
0.9138055237	0.9287439032	0.927746708	0.9113204239	0.9209743957	0.9259910449	0.9217253386	0.9202326456	0.9212446397	0.9252277506	0.9267415341	0.9159931954	0.928972992	0.9174717547	0.9199538934	0.9269999817	0.921732972	0.920499682
0.8036685378	0.8294861884	0.8147498958	0.8332402007	0.8343509158	0.8359630917	0.8085383272	0.7827775642	0.8399686339	0.845130741	0.8191644535	0.8352445528	0.815218948	0.8023505276	0.8088321908	0.8464968916	0.8391993377	0.8424842484
0.6842455299	0.7334477558	0.70399815	0.7499520447	0.7491472383	0.7446119468	0.6935365339	0.64898283	0.7578880838	0.7646327683	0.7189202759	0.7604496345	0.7060782614	0.6861764679	0.7022575367	0.7632482096	0.7531222253	0.7644009895
0.5611979809	0.6337977648	0.592977105	0.6641679403	0.6595254768	0.6552970388	0.5779614483	0.5153128039	0.6750086736	0.6871782988	0.6150790587	0.6821942052	0.5956953625	0.5716692185	0.5942974459	0.6792490177	0.6676908694	0.6815852004
0.4392415697	0.5375411252	0.4791442337	0.5768954609	0.5726946348	0.5594066768	0.4675158202	0.38215735	0.5878213084	0.6109749461	0.5138715524	0.6041518049	0.4864428159	0.4603146175	0.4858769382	0.6014975094	0.5878405449	0.5970874175
0.8554261865	0.7936051569	0.8426061302	0.8627859117	0.8231016982	0.8681847432	0.8229934097	0.8615630322	0.8658494495	0.8654367283	0.8729140899	0.8548272081	0.8534906974	0.7226925912	0.8378499152	0.8709403422	0.8159177127	0.8370098495
'''

def process_data_n(data_file):
    data_different_row = [row.split() for row in data_file.strip().split('\n')]
    data_matrix = np.expand_dims(np.array(data_different_row, dtype=float), axis=-1)
    data_matrix = np.concatenate((data_matrix[:, :3], data_matrix[:, 3:6], data_matrix[:, 6:9], data_matrix[:, 9:12], data_matrix[:, 12:15], data_matrix[:, 15:18]), axis=-1)

    return data_matrix

different_training_size_matrix = process_data_n(different_training_size)
print(f'create matrix of shape {different_training_size_matrix.shape}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = [
    ["SFT-4000", "CTL-4000"],
    ["SFT-5000", "CTL-5000"],
    ["SFT-5500", "CTL-5500"]
]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 10%"]

custom_palette = {
    'SFT': sns.color_palette("Set3")[1],
    'CTL': sns.color_palette("Set3")[2],
}

# Select only the specified cases
selected_cases = [1, 2, 4, 5]  # Indices for 90%, OOD 70%, OOD 50%, and OOD 10%
amazon_vis_matrix = different_training_size_matrix[selected_cases]

# Create a figure with subplots
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
sns.set(style="whitegrid")

margin = 0.005

for row in range(3):
    for col, case in enumerate(selected_cases):
        ax = axes[row, col]
        
        # Prepare data for plotting
        plot_data = [amazon_vis_matrix[col, :, 2*row + method] for method in range(2)]
        
        # Find min and max for this specific case
        case_min = np.min(amazon_vis_matrix[col, :, 2*row:2*row+2]) - margin
        case_max = np.max(amazon_vis_matrix[col, :, 2*row:2*row+2]) + margin
        
        # Create box plot for each case
        sns.boxplot(data=plot_data, palette=[custom_palette[model.split('-')[0]] for model in model_names[row]], width=0.6, ax=ax)
        
        # Customize each subplot
        ax.set_ylabel('F1 Score', fontsize=12)
        if row == 0:
            ax.set_title(case_labels[col], fontsize=14)
        ax.set_xticklabels(model_names[row], rotation=45, ha='right', fontsize=10)
        ax.tick_params(axis='y', labelsize=10)
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # Set y-axis limits to case-specific min and max
        ax.set_ylim(case_min, case_max)

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
different_sample_size = '''
0.9966249957	0.9803742024	0.9837499746	0.9811246788	0.97999902	0.9827498447	0.9814996658	0.9796239656	0.983749999																	
0.9202473632	0.925244912	0.9159879023	0.9279826978	0.925234818	0.9152289715	0.9252277506	0.9267415341	0.9159931954																	
0.8514748993	0.8211936648	0.8352359024	0.8418931197	0.8188728034	0.834703624	0.845130741	0.8191644535	0.8352445528																	
0.7819369798	0.7275088283	0.7586651557	0.7559467486	0.7190276855	0.7600909253	0.7646327683	0.7189202759	0.7604496345																	
0.7103954528	0.6282615124	0.6821441341	0.6749128113	0.6160401003	0.6842698327	0.6871782988	0.6150790587	0.6821942052																	
0.6454466575	0.5295260012	0.602807759	0.5959487276	0.5161262479	0.6050161448	0.6109749461	0.5138715524	0.6041518049																	
0.8665547613	0.8714377759	0.8545481765	0.8625843176	0.8696058411	0.8549724081	0.8654367283	0.8729140899	0.8548272081																	
'''

def process_data_n(data_file):
    data_different_row = [row.split() for row in data_file.strip().split('\n')]
    data_matrix = np.expand_dims(np.array(data_different_row, dtype=float), axis=-1)
    data_matrix = np.concatenate((data_matrix[:, :3], data_matrix[:, 3:6], data_matrix[:, 6:9]), axis=-1)

    return data_matrix

different_sample_size_matrix = process_data_n(different_sample_size)
print(f'create matrix of shape {different_sample_size_matrix.shape}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_names = ["CTL-1", "CTL-5", "CTL-20"]
case_labels = ["ID 90%", "OOD 70%", "OOD 50%", "OOD 10%"]

custom_palette = sns.color_palette("Set3")[:3]  # Use first 3 colors from Set3

# Select only the specified cases
selected_cases = [1, 2, 4, 5]  # Indices for 90%, OOD 70%, OOD 50%, and OOD 10%

# Create a figure with subplots
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
sns.set(style="whitegrid")

margin = 0.005

for i, case in enumerate(selected_cases):
    ax = axes[i]
    
    # Prepare data for plotting
    plot_data = [different_sample_size_matrix[case, :, method] for method in range(3)]
    
    # Find min and max for this specific case
    case_min = np.min(different_sample_size_matrix[case]) - margin
    case_max = np.max(different_sample_size_matrix[case]) + margin
    
    # Create box plot for each case
    sns.boxplot(data=plot_data, palette=custom_palette, width=0.6, ax=ax)
    
    # Customize each subplot
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title(case_labels[i], fontsize=14)
    ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Set y-axis limits to case-specific min and max
    ax.set_ylim(case_min, case_max)

# Adjust layout and show plot
plt.tight_layout()
plt.show()